# Part 1 · Notebook 03 — Order books and spreads

**Sessions:** S8 (order books, market structure) · S9 (spreads, impact) · **Clinic W3: microstructure data lab**

**You will:** measure the cost of walking an order book, compute quoted, effective and realized spreads from trades and quotes, see the intraday spread pattern, and estimate the spread from trade prices alone (Roll).

> The default data here is **synthetic** (generated with a fixed seed) so the notebook runs anywhere. For the graded lab, point `DATA_DIR` at the course trade/quote files.

In [ ]:
import sys, os
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))          # p1lib.py lives next to this notebook
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p1lib as p

p.use_course_style()
OUT_DIR = Path("outputs"); OUT_DIR.mkdir(exist_ok=True)   # CSV exports for Excel go here
pd.set_option("display.float_format", "{:,.4f}".format)
print("Offline fixtures:" if os.environ.get("P1_FIXTURES") else "Live data from FRED", os.environ.get("P1_FIXTURES", ""))

## 1. Walking the book

✏️ **Change me:** mid price, tick size and order sizes.

In [ ]:
MID, TICK = 100.00, 0.01                          # ✏️ Change me
bids, asks = p.synthetic_book(mid=MID, tick=TICK, levels=10, seed=0)
pd.concat({"bid price": bids["price"], "bid size": bids["size"],
           "ask price": asks["price"], "ask size": asks["size"]}, axis=1)

In [ ]:
ORDER_SIZES = [100, 1_000, 5_000, 10_000, 20_000, 40_000]    # ✏️ Change me
cost = pd.DataFrame([p.walk_the_book(asks, q, MID) for q in ORDER_SIZES]).set_index("qty")
ax = cost["cost_bps"].plot(marker="o", logx=True, title="Cost of a market buy vs order size (bps vs mid)")
ax.set_xlabel("shares (log scale)"); ax.set_ylabel("bps")
plt.show()
cost

## 2. Spreads from trades and quotes

- **Quoted** spread: ask − bid.
- **Effective** spread: 2 × |trade price − mid| (what the trader actually paid).
- **Realized** spread: measured against the mid 5 minutes later (what the liquidity provider kept); the difference is **price impact**.

In [ ]:
DATA_DIR = None     # ✏️ Change me: folder with course files, e.g. Path("data/microstructure")
TICKER = "SYNTH"    # ✏️ Change me

if DATA_DIR:
    quotes = pd.read_csv(Path(DATA_DIR) / f"{TICKER}_quotes.csv", parse_dates=["ts"])   # columns: ts, bid, ask
    trades = pd.read_csv(Path(DATA_DIR) / f"{TICKER}_trades.csv", parse_dates=["ts"])   # columns: ts, price, size[, side]
else:
    quotes, trades = p.simulate_trades_quotes(base_spread=0.02, seed=1)
    print("Using SYNTHETIC data (seeded).")
m = p.spread_measures(trades, quotes, realized_after="5min")
m[["ts", "price", "bid", "ask", "side", "quoted_bps", "effective_bps", "realized_bps", "impact_bps"]].head()

In [ ]:
m[["quoted_bps", "effective_bps", "realized_bps", "impact_bps"]].describe().T[["mean", "50%", "std"]]

## 3. The intraday pattern

In [ ]:
by_time = m.groupby(m["ts"].dt.tz_convert("America/New_York").dt.floor("30min").dt.strftime("%H:%M"))["quoted_bps"].mean()
ax = by_time.plot(marker="o", title="Average quoted spread by time of day (bps, New York time)")
ax.set_xlabel(""); ax.set_ylabel("bps")
plt.show()

## 4. Estimating the spread from trade prices only (Roll 1984)

Useful when you have prices but no quotes. Bid–ask bounce makes consecutive price changes negatively correlated.

In [ ]:
roll = p.roll_spread(trades.sort_values("ts")["price"].to_numpy())
avg_quoted = (quotes["ask"] - quotes["bid"]).mean()
print(f"Roll estimate: {roll:.4f}   average quoted spread: {avg_quoted:.4f}")

## 5. Compare several stocks (for your report)

✏️ With course data, list the tickers; with synthetic data, each 'stock' gets a different spread level.

In [ ]:
UNIVERSE = {"LIQUID": 0.01, "MIDCAP": 0.03, "SMALLCAP": 0.08, "ILLIQUID": 0.20, "PENNY": 0.02}   # ✏️ Change me
rows = []
for i, (name, base) in enumerate(UNIVERSE.items()):
    if DATA_DIR:
        q = pd.read_csv(Path(DATA_DIR) / f"{name}_quotes.csv", parse_dates=["ts"])
        t = pd.read_csv(Path(DATA_DIR) / f"{name}_trades.csv", parse_dates=["ts"])
    else:
        q, t = p.simulate_trades_quotes(base_spread=base, seed=10 + i)
    mm = p.spread_measures(t, q)
    rows.append({"stock": name, "quoted_bps": mm["quoted_bps"].mean(), "effective_bps": mm["effective_bps"].mean(),
                 "roll_estimate": p.roll_spread(t.sort_values("ts")["price"].to_numpy()),
                 "avg_quoted_spread": (q["ask"] - q["bid"]).mean()})
table = pd.DataFrame(rows).set_index("stock")
table.to_csv(OUT_DIR / "spread_comparison.csv")
p.log_research({"notebook": "03_spreads", "source": "synthetic" if not DATA_DIR else str(DATA_DIR)})
table

## Questions
1. Why does cost per share rise with order size? What does this imply for strategies that trade large size?
2. When is the effective spread smaller than the quoted spread in real data? (Hint: price improvement, midpoint executions.)
3. Why are spreads wider at the open? Relate it to adverse selection.
4. Write the microstructure section of your report.